## Shoe sales analysis

In [1]:
import pandas as pd
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
print('Imported successfully..!')

Imported successfully..!


In [2]:
df = pd.read_csv('shoes_sales_dataset.csv')
df.head(5)

,Sale_ID,Date,Brand,Shoe_Type,Color,Country,Sales_Channel,Price_$,Units_Sold,Revenue_USD
0,S1,2025-11-24,Nike,Boots,Blue,UK,Online,112.40,4,449.60
1,S2,2025-03-13,Skechers,Boots,Grey,USA,Mall,239.16,4,956.64
2,S3,2025-08-05,Nike,Running,White,UK,Mall,191.04,2,382.08
3,S4,2025-11-05,New Balance,Casual,Green,UAE,Mall,161.70,1,161.70
4,S5,2025-10-07,Adidas,Formal,Grey,France,Online,64.32,14,900.48


#### Data cleaning

In [3]:
def validating_df(df):
    print("="*60)
    print(" DATASET OVERVIEW ".center(60, "="))
    print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Duplicates: {df.duplicated().sum()}")
    print(f"Total Null Values: {df.isnull().sum().sum()}")
    
    print("\n" + " COLUMN INFORMATION ".center(60, "="))
    df.info(verbose=True, show_counts=True)
    
    print("\n" + " NULL VALUES PER COLUMN ".center(60, "="))
    print(df.isnull().sum())

    print("\n" + " NUMERIC DESCRIPTION ".center(60, "="))
    with pd.option_context('display.max_columns', None, 'display.expand_frame_repr', False):
        print(df.describe().T)

    print("="*60)
validating_df(df)


===================== DATASET OVERVIEW =====================
Shape: 1000 rows, 10 columns
Duplicates: 0
Total Null Values: 0

==================== COLUMN INFORMATION ====================
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Sale_ID        1000 non-null   str    
 1   Date           1000 non-null   str    
 2   Brand          1000 non-null   str    
 3   Shoe_Type      1000 non-null   str    
 4   Color          1000 non-null   str    
 5   Country        1000 non-null   str    
 6   Sales_Channel  1000 non-null   str    
 7   Price_$        1000 non-null   float64
 8   Units_Sold     1000 non-null   int64  
 9   Revenue_USD    1000 non-null   float64
dtypes: float64(2), int64(1), str(7)
memory usage: 78.3 KB

================== NULL VALUES PER COLUMN ==================
Sale_ID          0
Date             0
Brand            0
Shoe_Type   

In [4]:
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%B')
df['Quarter'] = df['Date'].dt.quarter
df['Price_Category'] = pd.cut(df['Price_$'], 
                                bins=[0, 75, 150, 300], 
                                labels=['Budget', 'Mid-Range', 'Premium'])
df['Profit_Margin_Est'] = np.where(df['Price_Category'] == 'Budget', 0.20,
                             np.where(df['Price_Category'] == 'Mid-Range', 0.35, 0.45))
df['Est_Profit'] = df['Revenue_USD'] * df['Profit_Margin_Est']

In [5]:
# Pivot table for heatmap
heatmap_data = df.pivot_table(values='Revenue_USD', 
                               index='Brand', 
                               columns='Shoe_Type', 
                               aggfunc='sum')

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale='RdYlGn',
    text=np.round(heatmap_data.values/1000, 1),
    texttemplate='%{text}K',
    textfont={"size": 10},
    colorbar_title="Revenue (USD)"
))

fig.update_layout(
    title='Revenue Heatmap: Brand vs Shoe Type',
    xaxis_title='Shoe Type',
    yaxis_title='Brand',
    height=500,
    template='plotly'
)
fig.show()

In [6]:
# Aggregate by brand and country
bubble_data = df.groupby(['Brand', 'Country']).agg({
    'Revenue_USD': 'sum',
    'Units_Sold': 'sum',
    'Price_$': 'mean'
}).reset_index()

fig = px.scatter(bubble_data, 
                 x='Units_Sold', 
                 y='Price_$', 
                 size='Revenue_USD',
                 color='Brand',
                 hover_name='Country',
                 animation_frame='Brand',  # Interactive slider
                 size_max=60,
                 title='Market Opportunity Matrix: Size=Revenue, Position=Price & Volume',
                 labels={'Units_Sold': 'Total Units Sold', 
                        'Price_$': 'Average Price (USD)'})
fig.update_layout(template='plotly_white', height=600)
fig.show()

In [7]:
import plotly.graph_objects as go

# Prepare nodes and links for Sankey
countries = df['Country'].unique()
channels = df['Sales_Channel'].unique()
types = df['Shoe_Type'].unique()

# Create flows
country_channel = df.groupby(['Country', 'Sales_Channel'])['Revenue_USD'].sum().reset_index()
channel_type = df.groupby(['Sales_Channel', 'Shoe_Type'])['Revenue_USD'].sum().reset_index()

# Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=list(countries) + list(channels) + list(types),
        color=['blue']*len(countries) + ['green']*len(channels) + ['red']*len(types)
    ),
    link=dict(
        source=list(range(len(countries))) * len(channels),
        target=[len(countries) + i for i in range(len(channels))] * len(countries),
        value=[1] * (len(countries) * len(channels))
    )
)])

# Better approach with actual flows
source_nodes = []
target_nodes = []
values = []
labels = list(countries) + list(channels) + list(types)

for _, row in country_channel.iterrows():
    source_nodes.append(list(countries).index(row['Country']))
    target_nodes.append(len(countries) + list(channels).index(row['Sales_Channel']))
    values.append(row['Revenue_USD'])

for _, row in channel_type.iterrows():
    source_nodes.append(len(countries) + list(channels).index(row['Sales_Channel']))
    target_nodes.append(len(countries) + len(channels) + list(types).index(row['Shoe_Type']))
    values.append(row['Revenue_USD'])

fig = go.Figure(data=[go.Sankey(
    node=dict(label=labels, pad=15, thickness=20),
    link=dict(source=source_nodes, target=target_nodes, value=values)
)])

fig.update_layout(title='Revenue Flow: Country → Channel → Shoe Type', 
                  font_size=10, height=600)
fig.show()

In [8]:
# Calculate metrics for each brand
brand_metrics = df.groupby('Brand').agg({
    'Revenue_USD': 'sum',
    'Units_Sold': 'sum',
    'Price_$': 'mean',
    'Sale_ID': 'count'
}).reset_index()

# Normalize metrics for radar chart
metrics = ['Revenue_USD', 'Units_Sold', 'Price_$', 'Sale_ID']
brand_metrics_norm = brand_metrics.copy()
for metric in metrics:
    brand_metrics_norm[metric] = (brand_metrics[metric] - brand_metrics[metric].min()) / \
                                  (brand_metrics[metric].max() - brand_metrics[metric].min())

fig = go.Figure()
for brand in brand_metrics_norm['Brand'].unique():
    brand_data = brand_metrics_norm[brand_metrics_norm['Brand'] == brand]
    fig.add_trace(go.Scatterpolar(
        r=brand_data[metrics].values[0],
        theta=['Revenue', 'Units Sold', 'Avg Price', 'Transactions'],
        fill='toself',
        name=brand
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Brand Performance Spider Chart',
    showlegend=True
)
fig.show()

In [9]:
# Daily revenue aggregation
daily_revenue = df.groupby('Date')['Revenue_USD'].sum().reset_index()
daily_revenue['MA7'] = daily_revenue['Revenue_USD'].rolling(window=7).mean()
daily_revenue['MA30'] = daily_revenue['Revenue_USD'].rolling(window=30).mean()

# Anomaly detection (outside 2 standard deviations)
mean_rev = daily_revenue['Revenue_USD'].mean()
std_rev = daily_revenue['Revenue_USD'].std()
daily_revenue['Anomaly'] = np.where(
    abs(daily_revenue['Revenue_USD'] - mean_rev) > 2*std_rev, 
    daily_revenue['Revenue_USD'], 
    np.nan
)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Revenue with Trends', 'Units Sold Volume'))

fig.add_trace(go.Scatter(x=daily_revenue['Date'], y=daily_revenue['Revenue_USD'],
                         mode='lines', name='Daily Revenue', line=dict(color='lightgrey')),
              row=1, col=1)
fig.add_trace(go.Scatter(x=daily_revenue['Date'], y=daily_revenue['MA7'],
                         mode='lines', name='7-Day MA', line=dict(color='blue', width=2)),
              row=1, col=1)
fig.add_trace(go.Scatter(x=daily_revenue['Date'], y=daily_revenue['MA30'],
                         mode='lines', name='30-Day MA', line=dict(color='red', width=2)),
              row=1, col=1)
fig.add_trace(go.Scatter(x=daily_revenue['Date'], y=daily_revenue['Anomaly'],
                         mode='markers', name='Anomaly', marker=dict(color='red', size=10)),
              row=1, col=1)

# Volume subplot
daily_units = df.groupby('Date')['Units_Sold'].sum().reset_index()
fig.add_trace(go.Bar(x=daily_units['Date'], y=daily_units['Units_Sold'],
                     name='Daily Units', marker_color='lightblue'),
              row=2, col=1)

fig.update_layout(height=800, title='Sales Time Series Analysis with Anomaly Detection',
                  hovermode='x unified')
fig.show()

In [10]:
product_perf = df.groupby(['Brand', 'Shoe_Type', 'Color']).agg({
    'Price_$': 'mean',
    'Units_Sold': 'sum',
    'Revenue_USD': 'sum'
}).reset_index()

fig = px.scatter_3d(product_perf, 
                     x='Price_$', 
                     y='Units_Sold', 
                     z='Revenue_USD',
                     color='Brand',
                     size='Revenue_USD',
                     hover_data=['Shoe_Type', 'Color'],
                     title='3D Product Performance: Price × Volume × Revenue',
                     labels={'Price_$': 'Avg Price', 
                            'Units_Sold': 'Total Units', 
                            'Revenue_USD': 'Total Revenue'})
fig.update_layout(scene=dict(
    xaxis_title='Average Price',
    yaxis_title='Units Sold',
    zaxis_title='Revenue'
))
fig.show()

In [11]:
fig = px.sunburst(df, 
                  path=['Sales_Channel', 'Country', 'Brand', 'Shoe_Type'], 
                  values='Revenue_USD',
                  color='Revenue_USD',
                  color_continuous_scale='Viridis',
                  title='Revenue Hierarchy: Channel → Country → Brand → Product Type')
fig.update_layout(height=700)
fig.show()

In [12]:
# Prepare normalized data
brand_summary = df.groupby(['Brand', 'Country']).agg({
    'Revenue_USD': 'sum',
    'Price_$': 'mean',
    'Units_Sold': 'sum',
    'Est_Profit': 'sum'
}).reset_index()

fig = px.parallel_coordinates(brand_summary, 
                               color='Revenue_USD',
                               dimensions=['Price_$', 'Units_Sold', 'Revenue_USD', 'Est_Profit'],
                               color_continuous_scale=px.colors.diverging.Tealrose,
                               title='Multi-Dimensional Brand Performance Comparison')
fig.show()

In [13]:
monthly_country = df.groupby(['Month', 'Month_Name', 'Country'])['Revenue_USD'].sum().reset_index()

fig = px.bar(monthly_country, 
             x='Country', 
             y='Revenue_USD',
             color='Country',
             animation_frame='Month_Name',
             range_y=[0, monthly_country['Revenue_USD'].max() * 1.1],
             title='Monthly Revenue by Country (Animated)')
fig.update_layout(height=500)
fig.show()

In [14]:
# Create correlation matrix
corr_data = df.pivot_table(values='Revenue_USD', 
                            index='Brand', 
                            columns='Shoe_Type', 
                            aggfunc='sum').fillna(0)

# Add clustering
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist

fig = px.imshow(corr_data.corr(),
                labels=dict(color="Correlation"),
                x=corr_data.columns,
                y=corr_data.columns,
                color_continuous_scale='RdBu_r',
                title='Shoe Type Revenue Correlation Matrix')
fig.update_layout(height=500, width=600)
fig.show()

In [15]:
channel_stats = df.groupby('Sales_Channel').agg({
    'Units_Sold': 'sum',
    'Revenue_USD': 'sum',
    'Est_Profit': 'sum'
}).reset_index()

fig = go.Figure()
fig.add_trace(go.Funnel(
    y=channel_stats['Sales_Channel'],
    x=channel_stats['Revenue_USD'],
    textposition="inside",
    textinfo="value+percent initial",
    marker=dict(color=["gold", "mediumturquoise", "darkorange"]),
))

fig.update_layout(title='Revenue Funnel by Sales Channel', height=400)
fig.show()

In [16]:
brand_revenue = df.groupby('Brand')['Revenue_USD'].sum().reset_index()
brand_revenue = brand_revenue.sort_values('Revenue_USD', ascending=False)

fig = go.Figure(go.Waterfall(
    name="Revenue",
    orientation="v",
    measure=["relative"] * len(brand_revenue) + ["total"],
    x=list(brand_revenue['Brand']) + ["Total"],
    textposition="outside",
    text=[f"${v:,.0f}" for v in brand_revenue['Revenue_USD']] + 
         [f"${brand_revenue['Revenue_USD'].sum():,.0f}"],
    y=list(brand_revenue['Revenue_USD']) + [brand_revenue['Revenue_USD'].sum()],
    connector={"line": {"color": "rgb(63, 63, 63)"}},
))

fig.update_layout(title="Revenue Contribution Waterfall by Brand",
                  showlegend=False, height=500)
fig.show()

In [17]:
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=3, cols=2,
    specs=[[{"type": "scatter"}, {"type": "pie"}],
           [{"type": "heatmap"}, {"type": "bar"}],
           [{"type": "scatter", "colspan": 2}, None]],
    subplot_titles=('Price vs Volume', 'Revenue by Channel', 
                    'Brand-Product Matrix', 'Top Countries',
                    'Daily Revenue Trend')
)

# 1. Scatter: Price vs Volume
price_vol = df.groupby('Brand').agg({'Price_$': 'mean', 'Units_Sold': 'sum'})
for brand in price_vol.index:
    fig.add_trace(go.Scatter(x=[price_vol.loc[brand, 'Price_$']], 
                            y=[price_vol.loc[brand, 'Units_Sold']],
                            mode='markers+text', name=brand,
                            text=[brand], textposition='top center',
                            marker=dict(size=15)),
                  row=1, col=1)

# 2. Pie: Channel Distribution
channel_rev = df.groupby('Sales_Channel')['Revenue_USD'].sum()
fig.add_trace(go.Pie(labels=channel_rev.index, values=channel_rev.values), row=1, col=2)

# 3. Heatmap: Brand-Product
heatmap = df.pivot_table(values='Revenue_USD', index='Brand', columns='Shoe_Type', aggfunc='sum')
fig.add_trace(go.Heatmap(z=heatmap.values, x=heatmap.columns, y=heatmap.index,
                         colorscale='Viridis'), row=2, col=1)

# 4. Bar: Top Countries
country_rev = df.groupby('Country')['Revenue_USD'].sum().sort_values(ascending=True)
fig.add_trace(go.Bar(x=country_rev.values, y=country_rev.index, orientation='h'),
              row=2, col=2)

# 5. Line: Daily Revenue
daily = df.groupby('Date')['Revenue_USD'].sum()
fig.add_trace(go.Scatter(x=daily.index, y=daily.values, mode='lines'), row=3, col=1)

fig.update_layout(height=1000, showlegend=False, 
                  title_text="Shoe Sales Executive Dashboard")
fig.show()

In [18]:
# RFM Analysis (using Sales_ID as proxy for customer)
rfm_data = df.groupby('Country').agg({
    'Date': lambda x: (df['Date'].max() - x.max()).days,  # Recency
    'Sale_ID': 'count',  # Frequency
    'Revenue_USD': 'sum'  # Monetary
}).reset_index()
rfm_data.columns = ['Country', 'Recency', 'Frequency', 'Monetary']

# Normalize and score
for col in ['Recency', 'Frequency', 'Monetary']:
    rfm_data[f'{col}_Score'] = pd.qcut(rfm_data[col], q=4, labels=[1,2,3,4])

fig = px.scatter_3d(rfm_data, x='Recency', y='Frequency', z='Monetary',
                    color='Country', size='Monetary',
                    title='Customer Market Segmentation (RFM Analysis)',
                    labels={'Recency': 'Days Since Last Purchase (Lower=Better)',
                           'Frequency': 'Number of Transactions',
                           'Monetary': 'Total Revenue'})
fig.update_layout(scene=dict(
    xaxis_title='Recency',
    yaxis_title='Frequency', 
    zaxis_title='Monetary Value'
))
fig.show()

In [20]:
df.columns

Index(['Sale_ID', 'Date', 'Brand', 'Shoe_Type', 'Color', 'Country',
       'Sales_Channel', 'Price_$', 'Units_Sold', 'Revenue_USD', 'Month',
       'Month_Name', 'Quarter', 'Price_Category', 'Profit_Margin_Est',
       'Est_Profit'],
      dtype='str')

In [21]:
# Simple Price Elasticity Analysis
df['Price_Bin'] = pd.qcut(df['Price_$'], q=5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

elasticity_data = df.groupby(['Shoe_Type', 'Price_Bin'], observed=False).agg(
    Units_Sold=('Units_Sold', 'sum'),
    Avg_Price=('Price_$', 'mean')
).reset_index()

# Create single plot with all shoe types
fig = go.Figure()

for shoe_type in df['Shoe_Type'].unique():
    type_data = elasticity_data[elasticity_data['Shoe_Type'] == shoe_type]
    
    fig.add_trace(go.Scatter(
        x=type_data['Avg_Price'], 
        y=type_data['Units_Sold'],
        mode='lines+markers',
        name=shoe_type,
        line=dict(width=3),
        marker=dict(size=10),
        text=[f"Price: ${p:.2f}<br>Units: {u}" for p, u in 
              zip(type_data['Avg_Price'], type_data['Units_Sold'])],
        hoverinfo='text+name'
    ))

fig.update_layout(
    title='Price Elasticity by Shoe Type',
    xaxis_title='Average Price (USD)',
    yaxis_title='Total Units Sold',
    height=600,
    template='plotly_white',
    hovermode='closest'
)
fig.show()

In [22]:
import calendar

# Create calendar heatmap data
df['Weekday'] = df['Date'].dt.day_name()
df['Week'] = df['Date'].dt.isocalendar().week
df['Month_Num'] = df['Date'].dt.month

calendar_data = df.groupby(['Month_Num', 'Weekday'])['Revenue_USD'].sum().reset_index()

# Order weekdays
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
calendar_data['Weekday'] = pd.Categorical(calendar_data['Weekday'], categories=weekday_order, ordered=True)

fig = px.density_heatmap(calendar_data, x='Weekday', y='Month_Num', z='Revenue_USD',
                         title='Revenue Calendar Heatmap: Day of Week vs Month',
                         color_continuous_scale='RdYlGn',
                         labels={'Month_Num': 'Month', 'Weekday': 'Day of Week'})
fig.update_layout(height=500)
fig.show()

In [23]:
# Track product performance over time
product_lifecycle = df.groupby(['Date', 'Shoe_Type'])['Revenue_USD'].sum().reset_index()
product_lifecycle['Cumulative_Revenue'] = product_lifecycle.groupby('Shoe_Type')['Revenue_USD'].cumsum()
product_lifecycle['Moving_Avg_7'] = product_lifecycle.groupby('Shoe_Type')['Revenue_USD'].transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)

fig = make_subplots(rows=2, cols=1, 
                    subplot_titles=('Daily Revenue by Product Type', 'Cumulative Revenue Growth'))

for shoe_type in product_lifecycle['Shoe_Type'].unique():
    data = product_lifecycle[product_lifecycle['Shoe_Type'] == shoe_type]
    fig.add_trace(
        go.Scatter(x=data['Date'], y=data['Moving_Avg_7'], 
                  name=shoe_type, mode='lines'),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=data['Date'], y=data['Cumulative_Revenue'],
                  name=shoe_type, mode='lines'),
        row=2, col=1
    )

fig.update_layout(height=800, title='Product Lifecycle & Growth Analysis',
                  hovermode='x unified')
fig.show()

In [24]:
# Find product combinations frequently sold together (by date and country)
from itertools import combinations

# Get products sold on same date in same country
basket_data = df.groupby(['Date', 'Country'])['Shoe_Type'].apply(list).reset_index()

# Find combinations
combos = []
for products in basket_data['Shoe_Type']:
    if len(products) > 1:
        combos.extend(list(combinations(sorted(set(products)), 2)))

combo_df = pd.DataFrame(combos, columns=['Product_A', 'Product_B'])
combo_counts = combo_df.groupby(['Product_A', 'Product_B']).size().reset_index(name='Count')

# Network graph
import networkx as nx

G = nx.Graph()
for _, row in combo_counts.iterrows():
    G.add_edge(row['Product_A'], row['Product_B'], weight=row['Count'])

pos = nx.spring_layout(G, k=3, iterations=50)
edge_x = []
edge_y = []
for edge in G.edges(data=True):
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines',
                        line=dict(width=1, color='#888'),
                        hoverinfo='none')

node_x = [pos[node][0] for node in G.nodes()]
node_y = [pos[node][1] for node in G.nodes()]
node_text = list(G.nodes())

node_trace = go.Scatter(x=node_x, y=node_y, mode='markers+text',
                        text=node_text, textposition='top center',
                        marker=dict(size=[G.degree(node)*10 for node in G.nodes()],
                                   color='lightblue'),
                        hoverinfo='text')

fig = go.Figure(data=[edge_trace, node_trace],
                layout=go.Layout(title='Market Basket: Frequently Sold Together',
                               showlegend=False, hovermode='closest'))
fig.show()

In [26]:
# Boston Matrix style analysis
brand_perf = df.groupby('Brand').agg({
    'Revenue_USD': 'sum',
    'Units_Sold': 'sum',
    'Price_$': 'mean'
}).reset_index()

# Calculate market share and growth
brand_perf['Market_Share'] = brand_perf['Revenue_USD'] / brand_perf['Revenue_USD'].sum() * 100
brand_perf['Avg_Revenue_Per_Unit'] = brand_perf['Revenue_USD'] / brand_perf['Units_Sold']

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=brand_perf['Market_Share'],
    y=brand_perf['Avg_Revenue_Per_Unit'],
    mode='markers+text',
    marker=dict(
        size=brand_perf['Revenue_USD']/1000,
        color=brand_perf['Price_$'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Avg Price")
    ),
    text=brand_perf['Brand'],
    textposition='top center'
))

# Add quadrant lines
avg_share = brand_perf['Market_Share'].mean()
avg_rev_unit = brand_perf['Avg_Revenue_Per_Unit'].mean()

fig.add_hline(y=avg_rev_unit, line_dash="dash", line_color="red")
fig.add_vline(x=avg_share, line_dash="dash", line_color="red")

fig.update_layout(
    title='Brand Performance Matrix (Bubble Size = Revenue)',
    xaxis_title='Market Share (%)',
    yaxis_title='Revenue Per Unit ($)',
    annotations=[
        dict(x=avg_share/2, y=avg_rev_unit*1.5, text="Stars", showarrow=False),
        dict(x=avg_share*1.5, y=avg_rev_unit*1.5, text="Cash Cows", showarrow=False),
        dict(x=avg_share/2, y=avg_rev_unit/2, text="Question Marks", showarrow=False),
        dict(x=avg_share*1.5, y=avg_rev_unit/2, text="Dogs", showarrow=False)
    ]
)
fig.show()

In [28]:
# Calculate rolling metrics
daily_metrics = df.groupby('Date').agg({
    'Revenue_USD': 'sum',
    'Units_Sold': 'sum',
    'Price_$': 'mean'
}).reset_index()

# Rolling windows
daily_metrics['Rev_Rolling_7'] = daily_metrics['Revenue_USD'].rolling(7).mean()
daily_metrics['Rev_Rolling_30'] = daily_metrics['Revenue_USD'].rolling(30).mean()
daily_metrics['Volatility'] = daily_metrics['Revenue_USD'].rolling(7).std()
daily_metrics['Efficiency'] = daily_metrics['Revenue_USD'] / daily_metrics['Units_Sold']

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Revenue with Bollinger Bands', 'Volatility', 'Efficiency (Revenue per Unit)'),
    shared_xaxes=True,
    vertical_spacing=0.1
)

# Bollinger Bands
fig.add_trace(go.Scatter(x=daily_metrics['Date'], y=daily_metrics['Revenue_USD'],
                        name='Daily Revenue', line=dict(color='gray', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=daily_metrics['Date'], y=daily_metrics['Rev_Rolling_7'],
                        name='7-day MA', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=daily_metrics['Date'], 
                        y=daily_metrics['Rev_Rolling_7'] + 2*daily_metrics['Volatility'],
                        name='Upper Band', line=dict(dash='dash', color='red')), row=1, col=1)
fig.add_trace(go.Scatter(x=daily_metrics['Date'], 
                        y=daily_metrics['Rev_Rolling_7'] - 2*daily_metrics['Volatility'],
                        name='Lower Band', line=dict(dash='dash', color='red'),
                        fill='tonexty', fillcolor='rgba(255,0,0,0.1)'), row=1, col=1)

# Volatility
fig.add_trace(go.Scatter(x=daily_metrics['Date'], y=daily_metrics['Volatility'],
                        name='Volatility', line=dict(color='orange')), row=2, col=1)

# Efficiency
fig.add_trace(go.Scatter(x=daily_metrics['Date'], y=daily_metrics['Efficiency'],
                        name='Efficiency', line=dict(color='green')), row=3, col=1)

fig.update_layout(height=900, title='Advanced Performance Metrics Dashboard',
                  hovermode='x unified')
fig.show()

In [29]:
# Prepare geo data with time dimension
geo_time = df.groupby(['Date', 'Country']).agg({
    'Revenue_USD': 'sum',
    'Units_Sold': 'sum',
    'Price_$': 'mean'
}).reset_index()

# Create animated choropleth
fig = px.choropleth(geo_time,
                    locations='Country',
                    locationmode='country names',
                    color='Revenue_USD',
                    hover_name='Country',
                    animation_frame=geo_time['Date'].dt.strftime('%Y-%m-%d'),
                    color_continuous_scale='Viridis',
                    title='Animated Global Sales Distribution',
                    labels={'Revenue_USD': 'Revenue (USD)'})

fig.update_layout(height=600, 
                  geo=dict(showframe=False, 
                          showcoastlines=True,
                          projection_type='equirectangular'))
fig.show()

C:\Users\venka\AppData\Local\Temp\ipykernel_21312\468916243.py:9: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(geo_time,


In [30]:
# Create gauge for KPI monitoring
total_revenue = df['Revenue_USD'].sum()
avg_revenue = df['Revenue_USD'].mean()
target_revenue = total_revenue * 1.2  # 20% growth target

fig = go.Figure()

fig.add_trace(go.Indicator(
    mode="gauge+number+delta",
    value=total_revenue,
    delta={'reference': total_revenue * 0.8, 'increasing': {'color': "green"}},
    domain={'x': [0, 1], 'y': [0, 1]},
    title={'text': "Total Revenue vs Target"},
    gauge={
        'axis': {'range': [None, target_revenue]},
        'bar': {'color': "darkblue"},
        'steps': [
            {'range': [0, target_revenue * 0.5], 'color': "lightgray"},
            {'range': [target_revenue * 0.5, target_revenue * 0.8], 'color': "lightgreen"},
            {'range': [target_revenue * 0.8, target_revenue], 'color': "green"}
        ],
        'threshold': {
            'line': {'color': "red", 'width': 4},
            'thickness': 0.75,
            'value': total_revenue
        }
    }
))

fig.update_layout(height=400, title='Revenue Performance Gauge')
fig.show()

In [31]:
# Create affinity matrix between products and colors
affinity_data = df.pivot_table(
    values='Revenue_USD',
    index='Shoe_Type',
    columns='Color',
    aggfunc='sum'
).fillna(0)

# Normalize for better visualization
affinity_normalized = affinity_data.div(affinity_data.sum(axis=1), axis=0)

# Create annotated heatmap
fig = px.imshow(affinity_normalized,
                labels=dict(x="Color", y="Shoe Type", color="Revenue Share"),
                x=affinity_normalized.columns,
                y=affinity_normalized.index,
                color_continuous_scale='YlOrRd',
                title='Product-Color Affinity Matrix',
                text_auto='.1%')

fig.update_layout(height=500)
fig.show()

In [32]:
# Prepare racing bar chart data
monthly_brand = df.groupby(['Month_Name', 'Brand'])['Revenue_USD'].sum().reset_index()
monthly_brand['Month_Num'] = pd.to_datetime(monthly_brand['Month_Name'], format='%B').dt.month
monthly_brand = monthly_brand.sort_values('Month_Num')

# Create animated bar chart
fig = px.bar(monthly_brand,
             x='Brand',
             y='Revenue_USD',
             color='Brand',
             animation_frame='Month_Name',
             range_y=[0, monthly_brand['Revenue_USD'].max() * 1.1],
             title='Monthly Brand Revenue Race',
             text='Revenue_USD')

fig.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig.update_layout(height=500, xaxis={'categoryorder':'total descending'})
fig.show()

In [34]:
# Create violin plots for price distribution
fig = make_subplots(rows=2, cols=1, 
                    subplot_titles=('Price Distribution by Brand', 
                                  'Revenue Distribution by Sales Channel'))

# Violin plots for prices
for brand in df['Brand'].unique():
    fig.add_trace(
        go.Violin(y=df[df['Brand'] == brand]['Price_$'],
                 name=brand,
                 box_visible=True,
                 meanline_visible=True),
        row=1, col=1
    )

# Revenue distribution
for channel in df['Sales_Channel'].unique():
    fig.add_trace(
        go.Box(y=df[df['Sales_Channel'] == channel]['Revenue_USD'],
               name=channel,
               boxmean='sd'),
        row=2, col=1
    )

fig.update_layout(height=800, showlegend=True,
                  title='Statistical Distribution Analysis')
fig.show()

In [36]:
# Generate automated insights
insights = []

# Top performer
top_brand = df.groupby('Brand')['Revenue_USD'].sum().idxmax()
top_revenue = df.groupby('Brand')['Revenue_USD'].sum().max()
insights.append(f"🏆 {top_brand} leads with ${top_revenue:,.0f} revenue")

# Most efficient channel
channel_efficiency = df.groupby('Sales_Channel').agg({
    'Revenue_USD': 'sum',
    'Sale_ID': 'count'
})
channel_efficiency['Rev_Per_Sale'] = channel_efficiency['Revenue_USD'] / channel_efficiency['Sale_ID']
best_channel = channel_efficiency['Rev_Per_Sale'].idxmax()
insights.append(f"📈 {best_channel} has highest revenue per transaction")

# Seasonal insight
df['Month'] = df['Date'].dt.month
monthly_rev = df.groupby('Month')['Revenue_USD'].sum()
best_month = monthly_rev.idxmax()
insights.append(f"📅 Month {best_month} shows peak sales period")

# Product opportunity
avg_price = df.groupby('Shoe_Type')['Price_$'].mean()
high_margin_type = avg_price.idxmax()
insights.append(f"💎 {high_margin_type} shoes command premium pricing")

# Create interactive insights display
fig = go.Figure()

for i, insight in enumerate(insights):
    fig.add_annotation(
        x=0.5,
        y=1 - (i * 0.2),
        text=insight,
        showarrow=False,
        font=dict(size=14),
        xref="paper",
        yref="paper"
    )

fig.update_layout(
    title="🔍 Automated Business Insights",
    height=400,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
)
fig.show()